In [0]:
%sql

USE CATALOG `abdullah-de-airbnb`;
USE SCHEMA default;

In [0]:
%sql
WITH with_rn AS (
SELECT
    *,
    row_number() OVER(PARTITION BY id ORDER BY id) AS rn
  FROM
    airbnb_reviewers_silver
)
  SELECT *
  FROM with_rn
  WHERE rn <> 1

In [0]:
%sql
SELECT 
    host_id,
    FIRST(h.host_name) AS name,
    COUNT(*) AS listing_count
FROM airbnb_hosts_silver h
JOIN airbnb_listings_bronze l
ON h.id = l.host_id
GROUP BY host_id
ORDER BY listing_count DESC

In [0]:
%sql

SELECT
  h.id as id,
  FIRST(h.host_name) as name,
  CONCAT("$", SUM(l.estimated_revenue)) as total_revenue
FROM airbnb_hosts_silver h
JOIN airbnb_listings_silver l
ON h.id = l.host_id
WHERE l.estimated_revenue IS NOT NULL
GROUP BY h.id
ORDER BY SUM(l.estimated_revenue) DESC

In [0]:
%sql

SELECT
  h.id as id,
  FIRST(h.host_name) as name,
  ROUND(SUM(l.reviews_per_month), 2) AS reviews_per_month,
  ROUND(AVG(l.review_scores_rating), 2) AS average_rating
FROM airbnb_hosts_silver h
JOIN airbnb_listings_silver l
ON h.id = l.host_id
WHERE l.review_scores_rating IS NOT NULL
GROUP BY h.id
ORDER BY AVG(l.review_scores_rating) DESC

In [0]:
%sql

SELECT
  c.country AS country,
  initcap(regexp_replace(trim(c.city), '[-_]+', ' ')) as city,
  ROUND(AVG(l.review_scores_rating), 2) AS average_rating
FROM airbnb_cities_silver c
LEFT JOIN airbnb_listings_silver l
ON c.city = l.city AND c.country = l.country
GROUP BY c.country, c.city
ORDER BY average_rating DESC

In [0]:
%sql
SELECT
  n.country as country,
  n.city as city,
  n.neighbourhood_group as neighbourhood_group,
  n.neighbourhood as neighbourhood,
  ROUND(AVG(l.review_scores_rating), 2) as avg_reviews
FROM airbnb_neighbourhoods_silver n
LEFT JOIN airbnb_listings_silver l
  ON n.country = l.country
 AND n.city = l.city
 AND coalesce(n.neighbourhood_group, 'NULL') = coalesce(l.neighbourhood_group, 'NULL')
 AND n.neighbourhood = l.neighbourhood
GROUP BY
  n.country,
  n.city,
  n.neighbourhood_group,
  n.neighbourhood
ORDER BY AVG(l.review_scores_rating) DESC


In [0]:
%sql

SELECT
  h.id AS host_id,
  FIRST(h.host_name) AS host_name,
  l.id AS listing_id,
  FIRST(l.name) AS listing_name,
  ROUND(FIRST((
    SELECT AVG(review_scores_rating)
    FROM airbnb_listings_silver
    WHERE host_id = h.id
  )), 2) AS user_reviews_avg,
  ROUND(FIRST(review_scores_rating), 2) AS listing_review,
  ROUND(listing_review - user_reviews_avg, 2) AS skew
FROM airbnb_hosts_silver h
JOIN airbnb_listings_silver l
ON h.id = l.host_id
GROUP BY h.id, l.id
ORDER BY
  user_reviews_avg DESC,
  listing_review DESC,
  skew DESC